# 3D Anomaly Detection Benchmark — Anomaly-ShapeNet + Real3D-AD

**Setup:**
1. Enable the **T4 GPU** accelerator (Settings → Accelerator)
2. Attach the two datasets as Inputs (Add Input → Your Work → your uploads) —
   the roots are hard-coded in the *dataset* cell below
3. Set `REPO_URL` to your GitHub fork of this project
4. *Run All*

Expected wall-clock: Anomaly-ShapeNet ~30–60 min · Real3D-AD ~2–4 h.
For full-benchmark runs use *Save Version → Save & Run All* so everything
persists even if you disconnect.

<!-- RUN-BANNER -->
# ▶ CURRENT RUN: STEP 14 (alignment diagnostic on Anomaly-ShapeNet, ~20-30 min) — just press **Run All**
Only the cells of this step are active; every other code cell is commented out with `#`.
Active cells, in order:
- `%pip install -q open3d`
- `# ---- get the ad3d codebase (clone once, then always pull the latest)`
- `# ---- dataset roots (Kaggle inputs) + verification + loader sanity --`
- `# 14a: alignment diagnostic on 12 Anomaly-ShapeNet categories (normal `
- `# 14b: per-category summary of the alignment diagnostic. Never raises.`
- `# 14c: zip results + logs and show a download link (also in the Output`


In [ ]:
%pip install -q open3d
import torch
print('GPU available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# ---- get the ad3d codebase (clone once, then always pull the latest) -------
import os, glob, subprocess

REPO_URL = "https://github.com/nihal4/ad3d.git"

if not os.path.exists('ad3d'):
    os.system(f'git clone --depth 1 {REPO_URL}')
    if not os.path.exists('ad3d'):
        # fallback: the folder was uploaded as a Kaggle Dataset (Add Input)
        hits = glob.glob('/kaggle/input/**/run.py', recursive=True)
        assert hits, 'Could not clone REPO_URL and no uploaded ad3d folder found.'
        os.symlink(os.path.dirname(hits[0]), 'ad3d')
elif os.path.isdir('ad3d/.git'):
    print(subprocess.run(['git', '-C', 'ad3d', 'pull', '--ff-only'], capture_output=True, text=True).stdout)
print('code at:', os.path.abspath('ad3d'))
print(subprocess.run(['git', '-C', 'ad3d', 'log', '-1', '--format=%h %cd %s'], capture_output=True, text=True).stdout)


In [ ]:
# ---- dataset roots (Kaggle inputs) + verification + loader sanity -----------
import os, glob, sys

SHAPENET_ROOT = "/kaggle/input/datasets/smnihalahmed/anomaly-shapenet-v2/Anomaly-ShapeNet-v2/dataset/pcd"
REAL_ROOT     = "/kaggle/input/datasets/smnihalahmed/real3d-ad-pcd/Real3D-AD-PCD"

def verify(root, cat, name):
    n_cat = len([d for d in glob.glob(f"{root}/*/") if os.path.isdir(d)])
    tr = len(glob.glob(f"{root}/{cat}/train/*.pcd"))
    te = len(glob.glob(f"{root}/{cat}/test/*.pcd"))
    gt = len(glob.glob(f"{root}/{cat}/GT/*.txt")) or len(glob.glob(f"{root}/{cat}/gt/*.txt"))
    print(f"{name}: {n_cat} categories | {cat}: train={tr}  test={te}  gt={gt}")
    print(f"   -> {root}")
    assert n_cat > 0 and tr > 0 and te > 0 and gt > 0, f"{name} structure check failed"

verify(SHAPENET_ROOT, "ashtray0", "Anomaly-ShapeNet")
verify(REAL_ROOT, "airplane", "Real3D-AD")

# exercise the actual loaders on one category each (checks file formats too)
sys.path.insert(0, "ad3d/src")
from ad3d.datasets import load_train, load_test
for root, ds, cat in [(SHAPENET_ROOT, "shapenet", "ashtray0"),
                      (REAL_ROOT, "real3d", "airplane")]:
    tr = load_train(root, ds, cat)
    te = load_test(root, ds, cat)
    n_norm = sum(1 for s in te if s.label == 0)
    print(f"[loader] {ds}/{cat}: train={len(tr)}  test={len(te)} "
          f"({n_norm} normal / {len(te)-n_norm} anomalous)")

In [ ]:
# ---- smoke test: 2 categories, small settings (a few minutes) ---------------
# SMOKE = ('--num-group 512 --group-size 64 --max-nn 60 --n-scales 2 '
#          '--points-budget 30000')
# !python ad3d/run.py --dataset shapenet --data-root "{SHAPENET_ROOT}" --classes ashtray0,bowl0 {SMOKE} --tag smoke

In [ ]:
# SKIPPED: full Anomaly-ShapeNet benchmark - ALREADY COMPLETED (results already saved in earlier runs - uncomment to re-run)
# # ---- FULL Anomaly-ShapeNet benchmark (40 categories) -----------------------
# !python ad3d/run.py --dataset shapenet --data-root "{SHAPENET_ROOT}" --tag simple3dlite

In [ ]:
# SKIPPED: full Real3D-AD benchmark - ALREADY COMPLETED (results already saved in earlier runs - uncomment to re-run)
# # ---- FULL Real3D-AD benchmark (12 categories) -------------------------------
# !python ad3d/run.py --dataset real3d --data-root "{REAL_ROOT}" --tag simple3dlite

In [ ]:
# SKIPPED: ablation grid r1-r5 - ALREADY COMPLETED (see r1-r4 CSVs) (results already saved in earlier runs - uncomment to re-run)
# # ---- Real3D-AD ablation grid: alignment / cuts / topk -----------------------
# # Targets the 360-degree-train vs single-view-test gap. Each run is tagged and
# # compared against the simple3dlite baseline. Runtime note: --align icp adds a
# # RANSAC+ICP registration per test cloud (expect ~2x the baseline runtime).
# GRID = [
#     ("r1-cuts4",            "--cuts 4"),
#     ("r2-icp",              "--align icp"),
#     ("r3-icp-cuts4",        "--align icp --cuts 4"),
#     ("r4-icp-cuts4-topk32", "--align icp --cuts 4 --topk 32"),
#     ("r5-topk32",           "--topk 32"),
# ]
# for tag, extra in GRID:
#     print(f"\n########## {tag}: {extra} ##########", flush=True)
#     get_ipython().system(f'python ad3d/run.py --dataset real3d --data-root "{REAL_ROOT}" {extra} --tag {tag}')

In [ ]:
# ---- Real3D-AD phase 2 results so far ----------------------------------------
# r6-mhr6  (icp+cuts+align-poses 6):           O 0.689  P 0.913  P-AUPR 0.366
#   starfish 0.576 -> 0.686 (+11.0, aliasing fixed); airplane unchanged (thin wings)
# r7-vox003 (voxel 0.03, 3 cats):              airplane 0.580 -> 0.680 (+10.0!)
#   but starfish 0.686 -> 0.630 (fine voxel destabilises it) -> resolution
#   must be chosen PER CATEGORY.
#
# ---- r8: adaptive registration resolution ------------------------------------
# --align-voxel 0 auto-selects per category from {0.05, 0.03} using ONLY the
# training scans (quality of simulated-cut registration + stability penalty).
# The chosen voxel is printed per category and stored in the CSV. (~7-9h, T4)
# get_ipython().system('python ad3d/run.py --dataset real3d --data-root "{REAL_ROOT}" --align icp --cuts 4 --align-poses 6 --align-voxel 0 --tag r8-auto')

## Phase 3 — validity: multi-seed + diverse cuts (2×T4)
Uncomment one cell at a time. Each launches **one job per GPU** (`scripts/run_parallel.sh`); results land in `ad3d/results/`.
Order: 3a (seed-0 import) → 3b (base, seeds 0-2) → 3c (seeds 1-2) → 3d (diverse cuts) → 3e (aggregate).
Decision gate and rationale: see `NOVELTY_ROADMAP.md` ("Decision after r6").

In [ ]:
# 3a: import existing seed-0 runs so the aggregator can see them (r3 = icp-cuts4, r6 = mhr6)
# import glob, shutil, os
# os.makedirs('ad3d/results', exist_ok=True)
# for src, tag in [('r3-icp-cuts4_real3d_*.csv', 'icp-cuts4-s0'), ('r6-mhr6_real3d_*.csv', 'mhr6-s0')]:
#     f = glob.glob('ad3d/results_and_summary/results/' + src)[0]
#     shutil.copy(f, f'ad3d/results/{tag}_real3d_imported.csv')
#     print(tag, '<-', os.path.basename(f))


In [ ]:
# 3b: base (no alignment - fast) for seeds 0,1,2 (the clean seed-0 base run is not in the repo)
# !cd ad3d && CONFIGS="base" bash scripts/run_parallel.sh "{REAL_ROOT}" 0 1 2


In [ ]:
# 3c: seeds 1,2 for icp-cuts4 and mhr6 (seed 0 imported in 3a)
# !cd ad3d && CONFIGS="icp-cuts4 mhr6" bash scripts/run_parallel.sh "{REAL_ROOT}" 1 2


## RUN ORDER NOW (after 3a-3c are done)
**3g** feature-scale test (highest priority, ~2.5 h) -> **3h** compare -> **3f** ShapeNet seeds -> **3d** diverse-cuts probe (only if still worthwhile) -> **3e** aggregate.
Rationale: the Simple3D paper uses FPFH neighbourhoods 40/80/120, we use 100/200/300; see `NOVELTY_ROADMAP.md` (PRIORITY 0).

In [ ]:
# 3g: FEATURE-SCALE test (highest-priority experiment). The Simple3D paper uses FPFH neighbourhoods 40/80/120;
# our default is 100/200/300 (--max-nn 100). Run base at max-nn 40 and 60, seed 0, one per GPU (~2-2.5 h).
# Compare with base-s0 (0.637). If car/seahorse/chicken/airplane jump, re-run the aligned recipe at nn40 (mhr6-nn40).
# !cd ad3d && CONFIGS="base-nn40 base-nn60" bash scripts/run_parallel.sh "{REAL_ROOT}" 0


In [ ]:
# 3h: compare feature-scale runs with base-s0 and the published Simple3D per-category numbers. Never raises.
# (Simple3D numbers are from arXiv 2507.07435 as retrieved - verify in the PDF before citing.)
# import os, glob
# try:
#     import pandas as pd
#     RES = '/kaggle/working/ad3d/results' if os.path.isdir('/kaggle/working/ad3d/results') else 'ad3d/results'
#     def load(tag):
#         f = sorted(glob.glob(f'{RES}/{tag}_real3d_*.csv'))
#         return pd.read_csv(f[-1], index_col=0) if f else None
#     runs = {t: load(t) for t in ['base-s0', 'base-nn60-s0', 'base-nn40-s0']}
#     pub = {'car': (98.1, 99.2), 'seahorse': (93.0, 94.2), 'chicken': (82.6, 86.1),
#            'airplane': (76.5, 88.1), 'shell': (51.4, 71.6)}
#     for metric, k in (('O-AUROC', 'o_auroc'), ('P-AUROC', 'p_auroc')):
#         tab = pd.DataFrame({t: d[k] * 100 for t, d in runs.items() if d is not None}).round(1)
#         tab['Simple3D(paper)'] = [pub.get(c, (float('nan'),) * 2)[0 if k == 'o_auroc' else 1] for c in tab.index]
#         print(f'\n{metric} (x100); missing runs are simply absent')
#         print(tab.to_string())
# except Exception as e:
#     print('[!]', type(e).__name__, e)


## 3i-3j: RESULT OF 3g = feature scale matters (nn100 63.7 -> nn60 65.8 -> nn40 69.9, seed 0). Next:
**3i** finds the optimum scale (nn20, nn30; ~2.5 h) -> then **3j** re-runs the aligned recipe at the best scale, seed 0 (~4 h).
Car (50 vs 98) and seahorse (58 vs 93) are still far from Simple3D, so scale is not the whole story.


In [ ]:
# 3i: finer feature-scale sweep, seed 0, one config per GPU (~2-2.5 h). Uncomment to run.
# !cd ad3d && CONFIGS="base-nn30 base-nn20" bash scripts/run_parallel.sh "{REAL_ROOT}" 0


In [ ]:
# 3j: (a) does the scale keep improving below 20? base-nn10 (~1.5-2 h, GPU0) and
#     (b) does registration still add at the best scale so far? mhr6-nn20 (~4 h, GPU1). Seed 0, run together.
# !cd ad3d && CONFIGS="base-nn10 mhr6-nn20" bash scripts/run_parallel.sh "{REAL_ROOT}" 0


## 3k: RESULT OF 3j: base-nn10 74.4 (no floor yet), mhr6-nn20 75.1 (P-AUROC 92.3). Registration adds less at small scale (+2.6 vs +5).
Next: **mhr6-nn10** (best-scale + registration, ~4 h) together with **base-nn6** (is there a floor? note normals use 10 neighbours; ~1.5 h).

In [ ]:
# 3k: seed 0, one config per GPU.
# !cd ad3d && CONFIGS="base-nn6 mhr6-nn10" bash scripts/run_parallel.sh "{REAL_ROOT}" 0


## 3l: RESULT OF 3k: mhr6-nn10 = 78.3 O / 92.3 P-AUROC / 0.431 P-AUPR (seed 0); base-nn6 = 72.9 (worse than nn10 74.4 -> floor at nn10).
FINAL CONFIG = mhr6-nn10 (reference: base-nn10). Now confirm with seeds 1,2 (this cell, ~5-6 h; split across sessions if needed), then 3f (ShapeNet), then 3e.

## STEP 4 (PRIORITY, replaces the 3m screen): train on the published single-view CUT training data
Reading the official Simple3D code (github.com/hustCYQ/MiniShift-Simple3D, `data/real3d.py`) shows that its Real3D-AD numbers do **not**
use the 360-degree prototypes: it trains on `real3d_train_cut/<cls>/train_cut/*.asc`; the matching public download is GLFM's "Cut Training Data" (Real3D-mvtec, tiff format: `<cls>/train/good/xyz/*.tiff`)
(github.com/hustCYQ/GLFM-Multi-class-3DAD, Google Drive id `1l6jF5nrzgw-6EgjRjGw6071l1CyF_ep2`). It also scores objects by the MEAN
point score on Real3D-AD (not max), uses 4096 groups, a 5% coreset and an absolute voxel of 0.15. Test data is unchanged (official).

**4a** download + inventory (needs *Internet ON* in the notebook settings) -> **4b** run cut-nn10 and cut-mhr6-nn10 (seed 0) -> **4c** table.
The 3m screen cells are now low priority (keep for later).

In [ ]:
# 4a: download GLFM "Cut Training Data" (Real3D-mvtec, MVTec-3D tiff format) and inventory the TRAIN cuts. Never raises.
# Only train/good/xyz/*.tiff is used; the test set stays the OFFICIAL Real3D-AD one (this archive's test split is smaller).
# import os, sys, glob, subprocess, zipfile
# CUT_DIR = '/kaggle/working/real3d_cut'
# FILE_ID = '1l6jF5nrzgw-6EgjRjGw6071l1CyF_ep2'
# ARCH = '/kaggle/working/real3d_cut_download'
# CUT_ROOT = ''
# try:
#     os.makedirs(CUT_DIR, exist_ok=True)
#     if not os.listdir(CUT_DIR):                       # download + extract only once
#         subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=False)
#         import gdown
#         if not os.path.exists(ARCH):
#             gdown.download(id=FILE_ID, output=ARCH, quiet=False)
#         zipfile.ZipFile(ARCH).extractall(CUT_DIR)
#     subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tifffile'], check=False)
#     sys.path.insert(0, '/kaggle/working/ad3d/src')
#     from ad3d import datasets as D
#     import numpy as np, pandas as pd
#     cands = sorted({os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(p))))
#                     for p in glob.glob(f'{CUT_DIR}/**/train/good/xyz', recursive=True)})
#     CUT_ROOT = cands[0] if cands else ''
#     print('CUT_ROOT =', CUT_ROOT)
#     open('/kaggle/working/CUT_ROOT.txt', 'w').write(CUT_ROOT)
#     diag = lambda p: float(np.linalg.norm(p.max(0) - p.min(0)))
#     rows = {}
#     for cls in D.REAL3D_CLASSES:
#         try:
#             cuts = D.load_train_cut(CUT_ROOT, cls)
#             protos = D.load_train(REAL_ROOT, 'real3d', cls)
#             rows[cls] = {'n_cuts': len(cuts),
#                          'cut_pts_mean': int(np.mean([len(c) for _, c in cuts])),
#                          'proto_pts_mean': int(np.mean([len(p) for p in protos])),
#                          'cut_diag_mean': round(float(np.mean([diag(c) for _, c in cuts])), 2),
#                          'proto_diag_mean': round(float(np.mean([diag(p) for p in protos])), 2)}
#         except Exception as e:
#             rows[cls] = {'error': f'{type(e).__name__}: {e}'[:80]}
#     print(pd.DataFrame(rows).T.to_string())
#     print('\nExpected: n_cuts > 0 everywhere; cut_pts < proto_pts (a cut is part of an object);'
#           ' cut_diag a bit smaller than or similar to proto_diag (same units).')
# except Exception as e:
#     print('[!] download/inventory failed:', type(e).__name__, e)


In [ ]:
# 4b: train on the cut data, seed 0, one config per GPU. Run time depends on the number of cut files (see 4a).
#   cut-nn10      : Simple3D-like protocol (cuts only, no registration)
#   cut-mhr6-nn10 : cuts registered to the prototypes + multi-hypothesis registration of test clouds
# CUT_ROOT = open('/kaggle/working/CUT_ROOT.txt').read().strip()   # if the kernel was restarted
# !cd ad3d && CUT_ROOT="{CUT_ROOT}" CONFIGS="cut-nn10 cut-mhr6-nn10" bash scripts/run_parallel.sh "{REAL_ROOT}" 0


In [ ]:
# 4c: results table + object-score rules (max vs mean vs top-k), computed offline from the _scores.json files.
# !cd ad3d && python scripts/score_rules.py "results/cut-*-s0_real3d_*_scores.json"
# import os, glob
# try:
#     import pandas as pd
#     RES = '/kaggle/working/ad3d/results' if os.path.isdir('/kaggle/working/ad3d/results') else 'ad3d/results'
#     tags = ['base-nn10-s0', 'mhr6-nn10-s0', 'cut-nn10-s0', 'cut-mhr6-nn10-s0']
#     runs = {}
#     for t in tags:
#         f = sorted(glob.glob(f'{RES}/{t}_real3d_*[0-9].csv'))
#         if f:
#             runs[t] = pd.read_csv(f[-1], index_col=0)
#     for k in ('o_auroc', 'p_auroc', 'p_aupr', 'p_auroc_pooled'):
#         print(f'\n{k} (x100)'); print((pd.DataFrame({t: d[k] for t, d in runs.items()}) * 100).round(1).to_string())
# except Exception as e:
#     print('[!]', type(e).__name__, e)


## STEP 5: RESULT OF STEP 4: the cut data does NOT help (cut-nn10 66.9, cut-mhr6-nn10 74.2 vs ours 74.4 / 78.3).
But the object-score rule matters a lot (p99 / top200 / mean beat max by 2-5 pts). **5a** re-runs our best config so it
saves per-sample scores (deterministic: should reproduce 78.3) + tests adding the real cuts to our memory. Needs 4a (CUT_ROOT).

In [ ]:
# 5a: seed 0, one config per GPU (~1.5 h). Then run 5b.
# CUT_ROOT = open('/kaggle/working/CUT_ROOT.txt').read().strip()
# !cd ad3d && CUT_ROOT="{CUT_ROOT}" CONFIGS="mhr6-nn10 mhr6-nn10-pluscut" bash scripts/run_parallel.sh "{REAL_ROOT}" 0


In [ ]:
# 5b: object-score rules for both runs (offline, from the _scores.json files)
# !cd ad3d && python scripts/score_rules.py "results/mhr6-nn10*-s0_real3d_*_scores.json"


In [ ]:
# 5c: zip results + logs and show a download link (also in the Output tab). Never raises.
# import os, glob, zipfile, csv
# from IPython.display import FileLink, display
# BASE = "/kaggle/working/ad3d"
# OUT = "/kaggle/working/results_5.zip"
# PATTERNS = ("mhr6-nn10-s0", "mhr6-nn10-pluscut-s0")
# try:
#     n = 0
#     with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
#         for sub in ("results", "logs"):
#             for p in glob.glob(f"{BASE}/{sub}/**/*", recursive=True):
#                 if os.path.isfile(p):
#                     z.write(p, os.path.relpath(p, BASE)); n += 1
#     print(f"zipped {n} files -> {OUT} ({os.path.getsize(OUT)/1e6:.2f} MB)")
#     display(FileLink(OUT))
#     print("\n=== BEGIN SUMMARY ===")
#     for pat in PATTERNS:
#         for p in sorted(glob.glob(f"{BASE}/results/{pat}_real3d_*[0-9].csv")):
#             print("##", os.path.basename(p))
#             print(open(p).read())
#     print("=== END SUMMARY ===")
# except Exception as e:
#     print("[!]", type(e).__name__, e)


## STEP 6: FINAL CONFIG FROZEN -> seeds. RESULT OF STEP 5: mhr6-nn10-pluscut = 80.1 (max) / **81.4 (p99)**; mhr6-nn10 = 77.9 / 79.5.
Final = 360-degree prototypes + simulated cuts + published real cuts, RANSAC/ICP + MHR(6), max_nn 10, object score = 99th percentile.
6a runs seeds 1,2 for the final config and for mhr6-nn10-p99 (paired reference), then base-nn10-p99 seeds 0,1,2 (ablation row).
Seed 0 of the two registered configs is DERIVED offline from the step-5 score files (no re-run). ~4.5 h.

In [ ]:
# 6a: seeds for the frozen final config + references (7 jobs, round-robin over 2 GPUs, ~4.5 h)
# CUT_ROOT = open('/kaggle/working/CUT_ROOT.txt').read().strip()
# print('CUT_ROOT =', CUT_ROOT)
# !cd ad3d && CUT_ROOT="{CUT_ROOT}" CONFIGS="mhr6-nn10-pluscut-p99 mhr6-nn10-p99" bash scripts/run_parallel.sh "{REAL_ROOT}" 1 2
# !cd ad3d && CONFIGS="base-nn10-p99" bash scripts/run_parallel.sh "{REAL_ROOT}" 0 1 2


In [ ]:
# 6b: zip results + logs and show a download link (also in the Output tab). Never raises.
# import os, glob, zipfile
# from IPython.display import FileLink, display
# BASE = "/kaggle/working/ad3d"
# OUT = "/kaggle/working/results_6.zip"
# PATTERNS = ("mhr6-nn10-pluscut-p99-s", "mhr6-nn10-p99-s", "base-nn10-p99-s")
# try:
#     n = 0
#     with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
#         for sub in ("results", "logs"):
#             for p in glob.glob(f"{BASE}/{sub}/**/*", recursive=True):
#                 if os.path.isfile(p):
#                     z.write(p, os.path.relpath(p, BASE)); n += 1
#     print(f"zipped {n} files -> {OUT} ({os.path.getsize(OUT)/1e6:.2f} MB)")
#     display(FileLink(OUT))
#     print("\n=== BEGIN SUMMARY (mean rows) ===")
#     for pat in PATTERNS:
#         for p in sorted(glob.glob(f"{BASE}/results/{pat}*_real3d_*[0-9].csv")):
#             last = open(p).read().strip().splitlines()[-1]
#             print(os.path.basename(p), "|", last[:160])
#     print("=== END SUMMARY ===")
# except Exception as e:
#     print("[!]", type(e).__name__, e)


## STEP 7 (runs BEFORE step 6): multi-scale fusion test
Same final config (`pluscut-p99`) at three descriptor scales: nn10 (re-run, now with training reference stats), nn40, nn100.
The runs save per-sample scores + a training-only distance reference; fusion is then computed offline
(`scripts/fuse_scales.py`): each scale's p99 object score is z-normalised with its OWN training reference, then averaged with
equal weights. Pre-registered in RESULTS_LEDGER.md before any result. ~4 h (GPU0: nn10 then nn40, GPU1: nn100).

In [ ]:
# 7a: final config at three scales, seed 0 (job order puts nn10+nn40 on GPU0 and nn100 on GPU1, ~4 h)
# CUT_ROOT = open('/kaggle/working/CUT_ROOT.txt').read().strip()
# print('CUT_ROOT =', CUT_ROOT)
# !cd ad3d && CUT_ROOT="{CUT_ROOT}" CONFIGS="mhr6-nn10-pluscut-p99 mhr6-nn100-pluscut-p99 mhr6-nn40-pluscut-p99" bash scripts/run_parallel.sh "{REAL_ROOT}" 0


In [ ]:
# 7b: fusion table (offline, seconds). Primary = norm z; ratio/none shown for comparison.
# !cd ad3d && for n in z ratio none; do python scripts/fuse_scales.py --rule p99 --norm $n $(ls results/mhr6-nn10-pluscut-p99-s0_real3d_*[0-9].json results/mhr6-nn40-pluscut-p99-s0_real3d_*[0-9].json results/mhr6-nn100-pluscut-p99-s0_real3d_*[0-9].json); done


In [ ]:
# 7c: zip results + logs and show a download link (also in the Output tab). Never raises.
# import os, glob, zipfile
# from IPython.display import FileLink, display
# BASE = "/kaggle/working/ad3d"
# OUT = "/kaggle/working/results_7.zip"
# try:
#     n = 0
#     with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
#         for sub in ("results", "logs"):
#             for p in glob.glob(f"{BASE}/{sub}/**/*", recursive=True):
#                 if os.path.isfile(p):
#                     z.write(p, os.path.relpath(p, BASE)); n += 1
#     print(f"zipped {n} files -> {OUT} ({os.path.getsize(OUT)/1e6:.2f} MB)")
#     display(FileLink(OUT))
# except Exception as e:
#     print("[!]", type(e).__name__, e)


## STEP 8: one-at-a-time boost screen on the 5 WEAKEST categories (airplane, car, chicken, duck, shell)
**8 = Simple3D grouping**: final config + 4096 groups (was 2048) + 5% coreset (was 10%). Everything else unchanged.
Reference (final config `pluscut-p99`, seed 0) on these 5: O-AUROC 70.8 mean (airplane 74.1, car 69.7, chicken 75.1, duck 75.3, shell 59.7).
Gate (fixed now): 5-category O-AUROC mean >= 72.8 (+2.0) -> validate on all 12. Otherwise drop it and try higher resolution next.
Both GPUs split the 5 categories (~50 min).

In [ ]:
# 8a: Simple3D grouping on the 5 weakest categories, seed 0, categories split over both GPUs (~50 min)
# CUT_ROOT = open('/kaggle/working/CUT_ROOT.txt').read().strip()
# print('CUT_ROOT =', CUT_ROOT)
# !cd ad3d && SPLIT=1 CLASSES="car,airplane,duck,chicken,shell" CUT_ROOT="{CUT_ROOT}" CONFIGS="mhr6-nn10-pluscut-p99-g4096" bash scripts/run_parallel.sh "{REAL_ROOT}" 0


In [ ]:
# 8b: compare with the final config on the same 5 categories (reference numbers from seed 0). Never raises.
# import os, glob
# try:
#     import pandas as pd
#     RES = '/kaggle/working/ad3d/results'
#     ref = pd.DataFrame({'o_auroc': {'airplane': 74.1, 'car': 69.7, 'chicken': 75.1, 'duck': 75.3, 'shell': 59.7},
#                         'p_auroc': {'airplane': 88.4, 'car': 92.0, 'chicken': 89.0, 'duck': 98.3, 'shell': 90.2}})
#     f = sorted(glob.glob(f'{RES}/mhr6-nn10-pluscut-p99-g4096-s0_real3d_merged-*.csv'))
#     new = pd.read_csv(f[-1], index_col=0).drop('__mean__') * 100
#     t = pd.DataFrame({'O ref': ref.o_auroc, 'O g4096': new.o_auroc, 'P ref': ref.p_auroc, 'P g4096': new.p_auroc}).round(1)
#     t['dO'] = (t['O g4096'] - t['O ref']).round(1)
#     t.loc['MEAN'] = t.mean().round(2)
#     print(t.to_string())
#     m = t.loc['MEAN', 'O g4096']
#     print(f"\nGATE (5-cat O-AUROC mean >= 72.8): {'PASS -> validate on all 12' if m >= 72.8 else 'FAIL -> drop, try resolution next'}  ({m:.2f})")
# except Exception as e:
#     print('[!]', type(e).__name__, e)


In [ ]:
# 8c: zip results + logs and show a download link (also in the Output tab). Never raises.
# import os, glob, zipfile
# from IPython.display import FileLink, display
# BASE = "/kaggle/working/ad3d"
# OUT = "/kaggle/working/results_8.zip"
# try:
#     n = 0
#     with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
#         for sub in ("results", "logs"):
#             for p in glob.glob(f"{BASE}/{sub}/**/*", recursive=True):
#                 if os.path.isfile(p):
#                     z.write(p, os.path.relpath(p, BASE)); n += 1
#     print(f"zipped {n} files -> {OUT} ({os.path.getsize(OUT)/1e6:.2f} MB)")
#     display(FileLink(OUT))
# except Exception as e:
#     print("[!]", type(e).__name__, e)


## STEP 9: higher resolution (one change), 5 weakest categories. RESULT OF STEP 8: grouping FAILED (71.96 < 72.8).
Our voxel (0.01 in unit-normalised coordinates) keeps only 12k-42k points per cloud; Simple3D's absolute voxel 0.15 is
~0.0075 in our units (~2x our density). **9 = voxel 0.007 (~2x points) with max_nn 20**, i.e. the SAME physical neighbourhood
as our best nn10 at voxel 0.01 (twice the points per patch). Everything else = final config.
Reference on these 5: O-AUROC mean 70.78. Gate (fixed now): >= 72.8. ~1.5-2.5 h (both GPUs split the categories).

In [ ]:
# 9a: higher resolution on the 5 weakest categories, seed 0, categories split over both GPUs (~1.5-2.5 h)
# CUT_ROOT = open('/kaggle/working/CUT_ROOT.txt').read().strip()
# print('CUT_ROOT =', CUT_ROOT)
# !cd ad3d && SPLIT=1 CLASSES="car,airplane,duck,chicken,shell" CUT_ROOT="{CUT_ROOT}" CONFIGS="mhr6-nn20-pluscut-p99-vx007" bash scripts/run_parallel.sh "{REAL_ROOT}" 0


In [ ]:
# 9b: compare with the final config on the same 5 categories (reference = seed 0). Never raises.
# import os, glob
# try:
#     import pandas as pd
#     RES = '/kaggle/working/ad3d/results'
#     ref = pd.DataFrame({'o_auroc': {'airplane': 74.1, 'car': 69.7, 'chicken': 75.1, 'duck': 75.3, 'shell': 59.7},
#                         'p_auroc': {'airplane': 88.4, 'car': 92.0, 'chicken': 89.0, 'duck': 98.3, 'shell': 90.2}})
#     f = sorted(glob.glob(f'{RES}/mhr6-nn20-pluscut-p99-vx007-s0_real3d_merged-*.csv'))
#     new = pd.read_csv(f[-1], index_col=0).drop('__mean__') * 100
#     t = pd.DataFrame({'O ref': ref.o_auroc, 'O hires': new.o_auroc, 'P ref': ref.p_auroc, 'P hires': new.p_auroc}).round(1)
#     t['dO'] = (t['O hires'] - t['O ref']).round(1)
#     t.loc['MEAN'] = t.mean().round(2)
#     print(t.to_string())
#     m = t.loc['MEAN', 'O hires']
#     print(f"\nGATE (5-cat O-AUROC mean >= 72.8): {'PASS -> validate on all 12' if m >= 72.8 else 'FAIL -> drop'}  ({m:.2f})")
# except Exception as e:
#     print('[!]', type(e).__name__, e)


In [ ]:
# 9c: zip results + logs and show a download link (also in the Output tab). Never raises.
# import os, glob, zipfile
# from IPython.display import FileLink, display
# BASE = "/kaggle/working/ad3d"
# OUT = "/kaggle/working/results_9.zip"
# try:
#     n = 0
#     with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
#         for sub in ("results", "logs"):
#             for p in glob.glob(f"{BASE}/{sub}/**/*", recursive=True):
#                 if os.path.isfile(p):
#                     z.write(p, os.path.relpath(p, BASE)); n += 1
#     print(f"zipped {n} files -> {OUT} ({os.path.getsize(OUT)/1e6:.2f} MB)")
#     display(FileLink(OUT))
# except Exception as e:
#     print("[!]", type(e).__name__, e)


## STEP 10: NEW METHOD - Prototype Tolerance Field (PTF) + feature fusion, all 12 categories
Geometric channel: after registration, each test point's distance to the merged prototype surface, divided by the NORMAL
variation measured at that location between the 4 training prototypes (leave-one-out). Fused with our feature score:
point score = feature x (1 + u). One run also gives the ablation: features only, geometry only, geometry with ONE global
threshold (Template3D-AD-like), and that fused. Pre-registered in RESULTS_LEDGER.md. ~1.5-2 h (categories split over 2 GPUs).

In [ ]:
# 10a: final config + Prototype Tolerance Field fusion, all 12 categories, seed 0 (~1.5-2 h)
# CUT_ROOT = open('/kaggle/working/CUT_ROOT.txt').read().strip()
# print('CUT_ROOT =', CUT_ROOT)
# !cd ad3d && SPLIT=1 CUT_ROOT="{CUT_ROOT}" CONFIGS="mhr6-nn10-pluscut-p99-geo" bash scripts/run_parallel.sh "{REAL_ROOT}" 0


In [ ]:
# 10b: ablation table from the same run vs Template3D-AD (IJCAI 2025, verified per-category numbers). Never raises.
# import os, glob
# try:
#     import pandas as pd
#     RES = '/kaggle/working/ad3d/results'
#     t3d = {'airplane': 71.8, 'candybar': 87.5, 'car': 88.0, 'chicken': 78.6, 'diamond': 99.2, 'duck': 70.9,
#            'fish': 98.0, 'gemstone': 62.9, 'seahorse': 88.5, 'shell': 92.1, 'starfish': 82.9, 'toffees': 92.4}
#     f = sorted(glob.glob(f'{RES}/mhr6-nn10-pluscut-p99-geo-s0_real3d_merged-*.csv'))
#     d = pd.read_csv(f[-1], index_col=0)
#     cats = [c for c in d.index if c != '__mean__']
#     for metric in ('o_auroc', 'p_auroc', 'p_aupr'):
#         t = pd.DataFrame({'features': d[f'feat_{metric}'], 'geo(uniform)': d[f'geou_{metric}'],
#                           'geo(PTF)': d[f'geo_{metric}'], 'fused(uniform)': d[f'fuseu_{metric}'],
#                           'FUSED(PTF)': d[metric]}).loc[cats] * 100
#         if metric == 'o_auroc':
#             t['Template3D-AD'] = pd.Series(t3d)
#         t.loc['MEAN'] = t.mean()
#         print(f'\n{metric} (x100)'); print(t.round(1).to_string())
#     fused, feat = d.loc['__mean__', 'o_auroc'] * 100, d.loc['__mean__', 'feat_o_auroc'] * 100
#     print(f"\nGATE 1 (fused >= features + 1.5): {'PASS' if fused >= feat + 1.5 else 'FAIL'}  ({fused:.2f} vs {feat:.2f})")
#     print(f"GATE 2 (fused > Template3D-AD 84.4): {'PASS' if fused > 84.4 else 'FAIL'}  ({fused:.2f})")
# except Exception as e:
#     print('[!]', type(e).__name__, e)


In [ ]:
# 10c: zip results + logs and show a download link (also in the Output tab). Never raises.
# import os, glob, zipfile
# from IPython.display import FileLink, display
# BASE = "/kaggle/working/ad3d"
# OUT = "/kaggle/working/results_10.zip"
# try:
#     n = 0
#     with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
#         for sub in ("results", "logs"):
#             for p in glob.glob(f"{BASE}/{sub}/**/*", recursive=True):
#                 if os.path.isfile(p):
#                     z.write(p, os.path.relpath(p, BASE)); n += 1
#     print(f"zipped {n} files -> {OUT} ({os.path.getsize(OUT)/1e6:.2f} MB)")
#     display(FileLink(OUT))
# except Exception as e:
#     print("[!]", type(e).__name__, e)


## STEP 11: NEW METHOD - location-aware memory, all 12 categories (target: beat Template3D-AD 84.4 by >= 1 -> 85.4)
Each memory descriptor keeps the 3D position of its group centre in the registered frame; a test descriptor is compared only
with training descriptors from the SAME place on the object (radius rho = 0.10, fixed in advance). Same run also reports the
current global memory and rho = 0.05 / 0.15 (sensitivity). Pre-registered in RESULTS_LEDGER.md. ~1.5-2 h.

In [ ]:
# 11a: final config + location-aware memory, all 12 categories, seed 0, categories split over both GPUs (~1.5-2 h)
# CUT_ROOT = open('/kaggle/working/CUT_ROOT.txt').read().strip()
# print('CUT_ROOT =', CUT_ROOT)
# !cd ad3d && SPLIT=1 CUT_ROOT="{CUT_ROOT}" CONFIGS="mhr6-nn10-pluscut-p99-loc10" bash scripts/run_parallel.sh "{REAL_ROOT}" 0


In [ ]:
# 11b: global vs location-aware memory (same run) vs Template3D-AD (IJCAI 2025, verified). Never raises.
# import os, glob
# try:
#     import pandas as pd
#     RES = '/kaggle/working/ad3d/results'
#     t3d = {'airplane': 71.8, 'candybar': 87.5, 'car': 88.0, 'chicken': 78.6, 'diamond': 99.2, 'duck': 70.9,
#            'fish': 98.0, 'gemstone': 62.9, 'seahorse': 88.5, 'shell': 92.1, 'starfish': 82.9, 'toffees': 92.4}
#     f = sorted(glob.glob(f'{RES}/mhr6-nn10-pluscut-p99-loc10-s0_real3d_merged-*.csv'))
#     d = pd.read_csv(f[-1], index_col=0)
#     cats = [c for c in d.index if c != '__mean__']
#     for metric in ('o_auroc', 'p_auroc', 'p_aupr'):
#         t = pd.DataFrame({'global(now)': d[f'glob_{metric}'], 'loc rho=.05': d[f'loc05_{metric}'],
#                           'LOC rho=.10': d[f'loc10_{metric}'], 'loc rho=.15': d[f'loc15_{metric}']}).loc[cats] * 100
#         if metric == 'o_auroc':
#             t['Template3D-AD'] = pd.Series(t3d)
#         t.loc['MEAN'] = t.mean()
#         print(f'\n{metric} (x100)'); print(t.round(1).to_string())
#     loc, glob_ = d.loc['__mean__', 'loc10_o_auroc'] * 100, d.loc['__mean__', 'glob_o_auroc'] * 100
#     print(f"\nGATE 1 (loc >= global + 1.5): {'PASS' if loc >= glob_ + 1.5 else 'FAIL'}  ({loc:.2f} vs {glob_:.2f})")
#     print(f"GATE 2 (loc >= 85.4 = Template3D-AD + 1): {'PASS' if loc >= 85.4 else 'FAIL'}  ({loc:.2f})")
# except Exception as e:
#     print('[!]', type(e).__name__, e)


In [ ]:
# 11c: zip results + logs and show a download link (also in the Output tab). Never raises.
# import os, glob, zipfile
# from IPython.display import FileLink, display
# BASE = "/kaggle/working/ad3d"
# OUT = "/kaggle/working/results_11.zip"
# try:
#     n = 0
#     with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
#         for sub in ("results", "logs"):
#             for p in glob.glob(f"{BASE}/{sub}/**/*", recursive=True):
#                 if os.path.isfile(p):
#                     z.write(p, os.path.relpath(p, BASE)); n += 1
#     print(f"zipped {n} files -> {OUT} ({os.path.getsize(OUT)/1e6:.2f} MB)")
#     display(FileLink(OUT))
# except Exception as e:
#     print("[!]", type(e).__name__, e)


## STEP 12: seeds 1 and 2 for the NEW final config (location-aware memory). RESULT OF STEP 11: 85.6 (seed 0), passes both gates.
Each seed is one full 12-category run split over both GPUs (~1.5-2 h per seed, ~3.5-4 h total). 12b gives the 3-seed mean +/- std
(seed 0 values are built into the cell) and checks it against Template3D-AD (84.4) and the target (85.4).

In [ ]:
# 12a: seeds 1 and 2 for the new final config, each split over both GPUs (~3.5-4 h total)
# CUT_ROOT = open('/kaggle/working/CUT_ROOT.txt').read().strip()
# print('CUT_ROOT =', CUT_ROOT)
# !cd ad3d && SPLIT=1 CUT_ROOT="{CUT_ROOT}" CONFIGS="mhr6-nn10-pluscut-p99-loc10" bash scripts/run_parallel.sh "{REAL_ROOT}" 1 2


In [ ]:
# 12b: 3-seed summary (seed 0 from step 11 built in) vs Template3D-AD. Never raises.
# import os, glob
# try:
#     import numpy as np, pandas as pd
#     RES = '/kaggle/working/ad3d/results'
#     cats = ['airplane', 'candybar', 'car', 'chicken', 'diamond', 'duck', 'fish', 'gemstone', 'seahorse', 'shell', 'starfish', 'toffees']
#     s0 = {'loc10': [78.5, 100.0, 82.8, 72.7, 100.0, 89.7, 100.0, 78.7, 90.6, 56.3, 85.8, 92.3],
#           'glob': [73.9, 98.5, 69.3, 76.7, 98.3, 75.5, 85.5, 83.7, 81.1, 59.0, 89.3, 87.7]}
#     t3d = [71.8, 87.5, 88.0, 78.6, 99.2, 70.9, 98.0, 62.9, 88.5, 92.1, 82.9, 92.4]
#     runs = {'loc10': {0: pd.Series(s0['loc10'], index=cats)}, 'glob': {0: pd.Series(s0['glob'], index=cats)}}
#     extra = {}
#     for seed in (1, 2):
#         f = sorted(glob.glob(f'{RES}/mhr6-nn10-pluscut-p99-loc10-s{seed}_real3d_merged-*.csv'))
#         if not f:
#             print(f'[!] seed {seed} not found'); continue
#         d = pd.read_csv(f[-1], index_col=0).loc[cats]
#         runs['loc10'][seed] = d['loc10_o_auroc'] * 100
#         runs['glob'][seed] = d['glob_o_auroc'] * 100
#         extra[seed] = {k: d[k].mean() * 100 for k in ('p_auroc', 'p_aupr', 'p_auroc_pooled', 'p_aupr_pooled', 'loc05_o_auroc', 'loc15_o_auroc')}
#     for name, r in runs.items():
#         t = pd.DataFrame(r); t['mean'] = t.mean(axis=1); t['std'] = t.drop(columns='mean').std(axis=1)
#         if name == 'loc10':
#             t['Template3D-AD'] = t3d
#         print(f'\n{name} O-AUROC per seed (x100)'); print(t.round(1).to_string())
#         means = [t[c].mean() for c in r]
#         print(f'{name}: 12-category mean per seed = {[round(m, 2) for m in means]} -> {np.mean(means):.2f} +/- {np.std(means, ddof=1) if len(means) > 1 else 0:.2f}')
#     print('\nother metrics (seeds 1,2):', {s: {k: round(v, 2) for k, v in e.items()} for s, e in extra.items()})
#     m = np.mean([runs['loc10'][s].mean() for s in runs['loc10']])
#     print(f"\n3-seed mean {m:.2f}:  > Template3D-AD 84.4: {'YES' if m > 84.4 else 'NO'}   >= 85.4 (+1): {'YES' if m >= 85.4 else 'NO'}")
# except Exception as e:
#     print('[!]', type(e).__name__, e)


In [ ]:
# 12c: zip results + logs and show a download link (also in the Output tab). Never raises.
# import os, glob, zipfile
# from IPython.display import FileLink, display
# BASE = "/kaggle/working/ad3d"
# OUT = "/kaggle/working/results_12.zip"
# try:
#     n = 0
#     with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
#         for sub in ("results", "logs"):
#             for p in glob.glob(f"{BASE}/{sub}/**/*", recursive=True):
#                 if os.path.isfile(p):
#                     z.write(p, os.path.relpath(p, BASE)); n += 1
#     print(f"zipped {n} files -> {OUT} ({os.path.getsize(OUT)/1e6:.2f} MB)")
#     display(FileLink(OUT))
# except Exception as e:
#     print("[!]", type(e).__name__, e)


## STEP 13: Anomaly-ShapeNet with the FROZEN final method (held-out check). RESULT OF STEP 12: Real3D-AD 86.5 +/- 1.4 (3 seeds).
Same method as `mhr6-nn10-pluscut-p99-loc10`, minus the real cuts (they exist only for Real3D-AD) -> config `mhr6-nn10-p99-loc10`.
Nothing is tuned on Anomaly-ShapeNet. Seeds 0, 1, 2, one after the other; each seed is split over both GPUs by category and merged,
so every finished seed is already a complete 40-category result even if the session stops early. No cut download (4a) needed.


In [ ]:
# 13a: Anomaly-ShapeNet, frozen final method, seeds 0 1 2 (each split over both GPUs; time not yet measured, est. 1.5-2.5 h per seed)
# !cd ad3d && SPLIT=1 DATASET=shapenet CONFIGS="mhr6-nn10-p99-loc10" bash scripts/run_parallel.sh "{SHAPENET_ROOT}" 0 1 2


In [ ]:
# 13b: Anomaly-ShapeNet summary: location-aware vs global memory (same runs) + published methods. Never raises.
# import os, glob
# try:
#     import numpy as np, pandas as pd
#     RES = '/kaggle/working/ad3d/results'
#     per = {}
#     for seed in (0, 1, 2):
#         f = sorted(glob.glob(f'{RES}/mhr6-nn10-p99-loc10-s{seed}_shapenet_merged-*.csv'))
#         if not f:
#             print(f'[!] seed {seed} not found'); continue
#         per[seed] = pd.read_csv(f[-1], index_col=0).drop(index='__mean__', errors='ignore')
#     if per:
#         ch = ['loc10_o_auroc', 'glob_o_auroc', 'loc05_o_auroc', 'loc15_o_auroc', 'p_auroc', 'p_aupr', 'p_auroc_pooled', 'p_aupr_pooled', 'glob_p_auroc', 'glob_p_aupr']
#         rows = {s: {k: d[k].mean() * 100 for k in ch if k in d} for s, d in per.items()}
#         t = pd.DataFrame(rows).T
#         print(f'40-category means per seed (x100), n_categories = {[len(d) for d in per.values()]}')
#         print(t.round(2).to_string())
#         print('\nmean +/- std over seeds:')
#         for k in t.columns:
#             v = t[k].values
#             print(f'  {k:16s} {v.mean():6.2f} +/- {v.std(ddof=1) if len(v) > 1 else 0:.2f}')
#         loc = pd.concat([d['loc10_o_auroc'] for d in per.values()], axis=1).mean(axis=1) * 100
#         glo = pd.concat([d['glob_o_auroc'] for d in per.values()], axis=1).mean(axis=1) * 100
#         cat = pd.DataFrame({'loc10': loc, 'glob': glo, 'diff': loc - glo}).round(1)
#         print('\nper category O-AUROC, mean over seeds (x100):'); print(cat.to_string())
#         print(f"\nlocation-aware better on {(cat['diff'] > 0).sum()}/{len(cat)} categories, worse on {(cat['diff'] < 0).sum()}")
#         m = t['loc10_o_auroc'].mean()
#         print(f"\nPRIMARY loc10 O-AUROC {m:.2f}   global (same runs) {t['glob_o_auroc'].mean():.2f}   our old base nn100: 88.1")
#         print('published (Anomaly-ShapeNet O-AUROC): Template3D-AD 86.5, PASDF 90.0 (verified); Simple3D 86.0, Reg2Inv 86.1 (NOT verified)')
#         print(f"gate G1 (location-aware >= global, same runs): {'PASS' if m >= t['glob_o_auroc'].mean() else 'FAIL'}")
# except Exception as e:
#     print('[!]', type(e).__name__, e)


In [ ]:
# 13c: zip results + logs and show a download link (also in the Output tab). Never raises.
# import os, glob, zipfile
# from IPython.display import FileLink, display
# BASE = "/kaggle/working/ad3d"
# OUT = "/kaggle/working/results_13.zip"
# try:
#     n = 0
#     with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
#         for sub in ("results", "logs"):
#             for p in glob.glob(f"{BASE}/{sub}/**/*", recursive=True):
#                 if os.path.isfile(p):
#                     z.write(p, os.path.relpath(p, BASE)); n += 1
#     print(f"zipped {n} files -> {OUT} ({os.path.getsize(OUT)/1e6:.2f} MB)")
#     display(FileLink(OUT))
# except Exception as e:
#     print("[!]", type(e).__name__, e)


## STEP 14: alignment diagnostic (no tuning, ~20-30 min). RESULT OF STEP 13: Anomaly-ShapeNet loc 87.1 vs global 89.9 (G1 failed).
Question: do the categories where location-aware memory fails (caps, vase0, tap1, headset0) have POSE ERRORS on their normal
test objects (a), or are they well aligned and simply vary (b)? Registers the normal test clouds exactly like the frozen method
(seed 0) and measures the alignment at full resolution; also runs a fine ICP to see whether the pose was off.
6 failing + 6 control categories, split over the two GPUs. Send results_14.zip.


In [ ]:
# 14a: alignment diagnostic on 12 Anomaly-ShapeNet categories (normal test samples only), two processes in parallel
# (fixed: the old wait loop matched its own command line and never ended)
!cd ad3d && mkdir -p results logs && CUDA_VISIBLE_DEVICES=0 OMP_NUM_THREADS=2 python scripts/diag_align.py --data-root "{SHAPENET_ROOT}" --classes cap3,cap5,vase0,vase7,helmet1,bowl4 --out results/diag_align_part0.csv > logs/diag_align_part0.log 2>&1 & cd ad3d && CUDA_VISIBLE_DEVICES=1 OMP_NUM_THREADS=2 python scripts/diag_align.py --data-root "{SHAPENET_ROOT}" --classes cap4,tap1,headset0,bottle0,microphone0,bag0 --out results/diag_align_part1.csv 2>&1 | tee logs/diag_align_part1.log; wait; tail -3 logs/diag_align_part0.log


In [ ]:
# 14b: per-category summary of the alignment diagnostic. Never raises.
import glob
try:
    import pandas as pd
    fs = sorted(glob.glob('/kaggle/working/ad3d/results/diag_align_part*.csv'))
    d = pd.concat([pd.read_csv(f) for f in fs])
    fail = ['cap3', 'cap5', 'vase0', 'cap4', 'tap1', 'headset0']
    d['group'] = d['cls'].map(lambda c: 'FAILING' if c in fail else 'control')
    cols = ['res_med', 'res_p90', 'res_p99', 'rev_p90', 'ref_res_p90', 'ref_rot_deg', 'ref_trans', 'loo_med', 'loo_p90', 'fitness']
    print(f'{len(d)} normal test samples from {d.cls.nunique()} categories')
    print(d.groupby(['group', 'cls'])[cols].median().round(4).to_string())
    print(); print(d.groupby('group')[cols].median().round(4).to_string())
except Exception as e:
    print('[!]', type(e).__name__, e)


In [ ]:
# 14c: zip results + logs and show a download link (also in the Output tab). Never raises.
import os, glob, zipfile
from IPython.display import FileLink, display
BASE = "/kaggle/working/ad3d"
OUT = "/kaggle/working/results_14.zip"
try:
    n = 0
    with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
        for sub in ("results", "logs"):
            for p in glob.glob(f"{BASE}/{sub}/**/*", recursive=True):
                if os.path.isfile(p):
                    z.write(p, os.path.relpath(p, BASE)); n += 1
    print(f"zipped {n} files -> {OUT} ({os.path.getsize(OUT)/1e6:.2f} MB)")
    display(FileLink(OUT))
except Exception as e:
    print("[!]", type(e).__name__, e)


## NEW PLAN (replaces 3l/3f as the next step): improve FIRST, then seeds once on the frozen config.
**3m-A / 3m-B** screen 10 hyper-parameters on car, seahorse, chicken only (~25-40 min per config, 2 at a time). **3m-C** prints the table.
A winner must then be validated on all 12 categories (seed 0) before adoption. Seeds (3l) and ShapeNet (3f) come last, on the final config.

In [ ]:
# 3m-A: screen batch A, seed 0 (3 rounds, ~1.5-2 h). group size / number of groups / smoothing.
# !cd ad3d && PRESETS="g64 g256 ng1024 ng4096 sm6 sm24" bash scripts/screen.sh "{REAL_ROOT}"


In [ ]:
# 3m-B: screen batch B, seed 0 (2 rounds, ~1-1.5 h). coreset size / voxel size / object-score pooling.
# !cd ad3d && PRESETS="cs20 vx005 vx02 top32" bash scripts/screen.sh "{REAL_ROOT}"


In [ ]:
# 3m-C: table of the screen vs reference base-nn10 on car / seahorse / chicken. Never raises.
# import os, glob
# try:
#     import pandas as pd
#     RES = '/kaggle/working/ad3d/results' if os.path.isdir('/kaggle/working/ad3d/results') else 'ad3d/results'
#     cats = ['car', 'seahorse', 'chicken']
#     rows = {}
#     f = sorted(glob.glob(f'{RES}/base-nn10-s0_real3d_*.csv'))
#     if f:
#         d = pd.read_csv(f[-1], index_col=0); rows['REF base-nn10'] = d.loc[cats]
#     for f in sorted(glob.glob(f'{RES}/scr-*-s0_real3d_*.csv')):
#         d = pd.read_csv(f, index_col=0); rows[os.path.basename(f).split('_real3d')[0]] = d.loc[[c for c in cats if c in d.index]]
#     for k, name in (('o_auroc', 'O-AUROC'), ('p_auroc', 'P-AUROC'), ('p_aupr', 'P-AUPR')):
#         t = pd.DataFrame({n: r[k] for n, r in rows.items()}).T
#         t['mean'] = t.mean(axis=1)
#         print(f'\n{name}'); print((t * (100 if "auroc" in k else 1)).round(3 if k == "p_aupr" else 1).to_string())
# except Exception as e:
#     print('[!]', type(e).__name__, e)


In [ ]:
# 3l: seeds 1,2 for the final recipe and its no-registration reference (4 jobs, round-robin over 2 GPUs).
# If the session limit is a worry, run it in two parts:  CONFIGS="mhr6-nn10" ... 1 2   then   CONFIGS="base-nn10" ... 1 2
# !cd ad3d && CONFIGS="mhr6-nn10 base-nn10" bash scripts/run_parallel.sh "{REAL_ROOT}" 1 2


In [ ]:
# 3f: Anomaly-ShapeNet, seeds 0,1,2 for base (old scale, nn100) AND base-nn10 (new scale). Independent check: was the scale
# chosen on Real3D-AD test data also better on a different dataset? ~6 jobs, round-robin over the 2 GPUs.
# !cd ad3d && DATASET=shapenet CONFIGS="base base-nn10" bash scripts/run_parallel.sh "{SHAPENET_ROOT}" 0 1 2
# then:  !cd ad3d && python scripts/aggregate_seeds.py results/ --dataset shapenet


In [ ]:
# 3d: DIVERSE CUTS probe - only mhr6-div (the final recipe), seeds 0,1,2, split by category across both GPUs (~6 h)
# Gate: adopt only if paired mean gain over mhr6 >= +1.5 pts O-AUROC AND positive in 3/3 seeds
# !cd ad3d && SPLIT=1 CONFIGS="mhr6-div" bash scripts/run_parallel.sh "{REAL_ROOT}" 0 1 2


In [ ]:
# 3e: aggregate mean +/- std over seeds + paired deltas vs base (real3d). For shapenet: add  --dataset shapenet
# !cd ad3d && python scripts/aggregate_seeds.py results/


In [ ]:
# ---- summarize --------------------------------------------------------------
# import pandas as pd, glob
# frames = []
# for f in sorted(glob.glob('results/*.csv')):
#     df = pd.read_csv(f, index_col=0)
#     df.index.name = 'category'
#     df['source'] = f
#     frames.append(df)
# if frames:
#     full = pd.concat(frames)
#     means = full.groupby('source')[['o_auroc','p_auroc','p_aupr','p_auroc_pooled','p_aupr_pooled']].mean()
#     display(means)
#     full.drop(columns='source').to_csv('ad3d_results_summary.csv')
#     print('saved ad3d_results_summary.csv')
# else:
#     print('no full-run results yet')

## Next steps

Open **`ad3d/NOVELTY_ROADMAP.md`** — it maps the open failure modes in these
benchmarks (point-level AUPR, the gemstone/starfish/duck categories, the
360°-train vs single-view-test gap) to the exact functions to modify:

| To change... | Edit... |
|---|---|
| features / descriptors | `src/ad3d/features.py` → `compute_fpfh_ms()` |
| grouping / aggregation | `src/ad3d/features.py` → `lfsa()` |
| memory bank / distance | `src/ad3d/memory.py` |
| scoring rules | `src/ad3d/scoring.py` |
| whole pipeline | `src/ad3d/method.py` → subclass `Simple3DLite` |

Every run writes its full config + per-category results into `results/*.json`,
so tagged comparisons are always reproducible.